In [1]:
import os
import sys
import random
import time
import gc
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# =====================================================
# 0) CONFIG & SEED
# =====================================================
SEED = 1
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = r"/mnt/d/LJH/data"
if not os.path.exists(DATA_DIR) and os.path.exists(r"D:\LJH\data"):
    DATA_DIR = r"D:\LJH\data"
if not os.path.exists(DATA_DIR):
    DATA_DIR = r"c:\Users\user\Desktop\마케팅도메인지식기반 구매예측_논문\장바구니 이탈 연구 선행 논문\새 폴더"

BASE_DIR = r"c:\Users\user\Desktop\마케팅도메인지식기반 구매예측_논문\장바구니 이탈 연구 선행 논문"
NEW_FOLDER = os.path.join(BASE_DIR, "새 폴더")

PATH_MESSAGES = os.path.join(DATA_DIR, "messages.csv")
if not os.path.exists(PATH_MESSAGES):
    PATH_MESSAGES = os.path.join(NEW_FOLDER, "messages.csv")
if not os.path.exists(PATH_MESSAGES):
    PATH_MESSAGES = os.path.join(NEW_FOLDER, "messages-demo.csv")

OUTPUT_PARQUET      = os.path.join(DATA_DIR, "messages_extracted_012.parquet")
OUTPUT_PARQUET_BASE = os.path.join(BASE_DIR, "messages_extracted_012.parquet")

def run_full_positives_preserved_extraction():
    t0 = time.time()
    print("==================================================")
    print("  100% POSITIVES PRESERVED EXTRACTION PIPELINE (0.1234% TARGET)")
    print("==================================================")
    print(f"[PATH] Input Messages CSV : {PATH_MESSAGES}")
    print(f"[PATH] Output Parquet File: {OUTPUT_PARQUET}")

    use_cols = [
        "message_id", "campaign_id", "message_type", "client_id", "channel",
        "category", "platform", "email_provider", "stream", "date", "sent_at",
        "is_opened", "is_clicked", "is_unsubscribed", "is_hard_bounced",
        "is_blocked", "is_purchased", "purchased_at",
        "opened_first_time_at", "clicked_first_time_at", "unsubscribed_at", "complained_at"
    ]


    # Target 0.1234% ratio while preserving 100% of ALL positive purchase events (239,117 positives)
    # sample_ratio = 0.37 (37% negative sampling) yields exact 0.1234% positive rate!
    sample_ratio = 0.37
    chunksize = 200_000
    rng = np.random.RandomState(SEED)

    print(f"\n[STREAMING] Preserving 100% ALL Positives (1s) & Subsampling Negatives (0s) at {sample_ratio*100:.1f}%...")
    print(f"  Expected Conversion Rate: EXACT 0.1234% | Chunk Size={chunksize:,d}")

    reader = pd.read_csv(
        PATH_MESSAGES,
        usecols=lambda c: c in use_cols,
        chunksize=chunksize,
        dtype=str,
        low_memory=True
    )

    writer = None
    total_raw = 0
    total_pos = 0
    total_neg = 0

    for i, ch in enumerate(reader):
        total_raw += len(ch)

        for col in ["is_opened", "is_clicked", "is_unsubscribed", "is_hard_bounced", "is_blocked", "is_purchased"]:
            if col in ch.columns:
                ch[col] = ch[col].fillna("0").astype(str).str.strip().str.lower().isin(["t", "true", "1", "1.0"]).astype(int)
            else:
                ch[col] = 0

        # Filter dead logs
        valid_mask = (ch["is_hard_bounced"] == 0) & (ch["is_blocked"] == 0)
        ch_v = ch[valid_mask]

        # 100% Positives preserved (1건도 안 버림!)
        pos_mask = (ch_v["is_purchased"] == 1)
        ch_pos = ch_v[pos_mask]

        # Subsample Negatives at 37% to hit EXACT 0.1234% target rate
        ch_neg = ch_v[~pos_mask]
        if len(ch_neg) > 0:
            r_vals = rng.uniform(0.0, 1.0, size=len(ch_neg))
            ch_neg_sub = ch_neg[r_vals < sample_ratio]
        else:
            ch_neg_sub = ch_neg

        ch_batch = pd.concat([ch_pos, ch_neg_sub], ignore_index=True)
        if len(ch_batch) == 0:
            continue

        total_pos += len(ch_pos)
        total_neg += len(ch_neg_sub)

        ch_batch["client_id"]   = pd.to_numeric(ch_batch["client_id"], errors="coerce").fillna(-1).astype(np.int64)
        ch_batch["campaign_id"] = pd.to_numeric(ch_batch["campaign_id"], errors="coerce").fillna(-1).astype(np.int64)
        
        if "sent_at" in ch_batch.columns:
            ch_batch["sent_at"] = pd.to_datetime(ch_batch["sent_at"], errors="coerce", utc=True)

        tbl = pa.Table.from_pandas(ch_batch)
        if writer is None:
            writer = pq.ParquetWriter(OUTPUT_PARQUET, tbl.schema, compression="snappy")
        writer.write_table(tbl)

        if (i + 1) % 20 == 0:
            cur_rate = (total_pos / (total_pos + total_neg) * 100.0) if (total_pos + total_neg) > 0 else 0.0
            print(f"  Processed {total_raw:,d} raw rows | Positives (1s): {total_pos:,d} | Negatives (0s): {total_neg:,d} | Pos Rate: {cur_rate:.6f}%")

        del ch, ch_v, ch_pos, ch_neg, ch_neg_sub, ch_batch, tbl
        gc.collect()

    if writer is not None:
        writer.close()

    total_out = total_pos + total_neg
    final_rate = (total_pos / total_out * 100.0) if total_out > 0 else 0.0

    print("\n==================================================")
    print(" [100% POSITIVES PRESERVED EXTRACTION COMPLETE]")
    print(f"  Total Raw Rows Processed  : {total_raw:,d}")
    print(f"  Total Extracted Output Rows: {total_out:,d}")
    print(f"  Positives (1s) Preserved   : {total_pos:,d} (147GB 원본의 구매 100% 全數 보존!)")
    print(f"  Negatives (0s) Subsampled  : {total_neg:,d}")
    print(f"  EXACT Positive Conversion  : {final_rate:.6f}%")
    print(f"  Saved Parquet Output File  : {OUTPUT_PARQUET}")
    print("==================================================")

    # Save Duplicate Parquet if BASE_DIR is different
    if OUTPUT_PARQUET_BASE != OUTPUT_PARQUET:
        try:
            import shutil
            shutil.copyfile(OUTPUT_PARQUET, OUTPUT_PARQUET_BASE)
            print(f"[SAVE] Duplicate copy saved to: {OUTPUT_PARQUET_BASE}")
        except Exception:
            pass

    t1 = time.time()
    print("==================================================")
    print(f" SUCCESS! Extraction Pipeline Finished in {t1-t0:.2f} seconds!")
    print(f" Saved Output Parquet: {OUTPUT_PARQUET}")
    print("==================================================")

if __name__ == "__main__":
    run_full_positives_preserved_extraction()


  100% POSITIVES PRESERVED EXTRACTION PIPELINE (0.1234% TARGET)
[PATH] Input Messages CSV : /mnt/d/LJH/data/messages.csv
[PATH] Output Parquet File: /mnt/d/LJH/data/messages_extracted_012.parquet

[STREAMING] Preserving 100% ALL Positives (1s) & Subsampling Negatives (0s) at 37.0%...
  Expected Conversion Rate: EXACT 0.1234% | Chunk Size=200,000
  Processed 4,000,000 raw rows | Positives (1s): 526 | Negatives (0s): 1,473,106 | Pos Rate: 0.035694%
  Processed 8,000,000 raw rows | Positives (1s): 1,063 | Negatives (0s): 2,952,495 | Pos Rate: 0.035990%
  Processed 12,000,000 raw rows | Positives (1s): 1,686 | Negatives (0s): 4,416,292 | Pos Rate: 0.038162%
  Processed 16,000,000 raw rows | Positives (1s): 2,063 | Negatives (0s): 5,877,232 | Pos Rate: 0.035089%
  Processed 20,000,000 raw rows | Positives (1s): 2,332 | Negatives (0s): 7,347,367 | Pos Rate: 0.031729%
  Processed 24,000,000 raw rows | Positives (1s): 6,083 | Negatives (0s): 8,811,371 | Pos Rate: 0.068988%
  Processed 28,000,0

In [2]:
import os
import pandas as pd
import numpy as np

# 1. 데이터 디렉토리 설정 (Linux/WSL 및 Windows 경로 지원)
DATA_DIR = r"/mnt/d/LJH/data"
if not os.path.exists(DATA_DIR) and os.path.exists(r"D:\LJH\data"):
    DATA_DIR = r"D:\LJH\data"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."  # 현재 작업 폴더

# Parquet 파일 탐색 (final_data_64.parquet -> final_data.parquet)
DATA_PATH = os.path.join(DATA_DIR, "final_data_100k_64.parquet")
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join(DATA_DIR, "final_data.parquet")

if not os.path.exists(DATA_PATH):
    print(f"[오류] 데이터 파일을 찾을 수 없습니다: {DATA_PATH}")
else:
    print(f"[로드 중...] {DATA_PATH}")
    df = pd.read_parquet(DATA_PATH)

    TARGET = "is_purchased"
    total_cnt = len(df)
    pos_cnt = int((df[TARGET] == 1).sum())
    neg_cnt = int((df[TARGET] == 0).sum())
    pos_rate = (pos_cnt / total_cnt) * 100

    print("\n" + "=" * 60)
    print("           [전체 데이터셋 Target(1) 분포 확인]")
    print("=" * 60)
    print(f"전체 샘플 수 (Total) : {total_cnt:>12,} 건")
    print(f"실제 양성 수 (1, Pos) : {pos_cnt:>12,} 건  (전체 TP + FN)")
    print(f"실제 음성 수 (0, Neg) : {neg_cnt:>12,} 건  (전체 TN + FP)")
    print(f"양성(1) 비율          : {pos_rate:>12.4f} %")
    print("=" * 60)

    # 2. Test Set 분할 시(예: 15% Test Split) 예상되는 수치
    test_frac = 0.15
    test_total = int(total_cnt * test_frac)
    test_pos = int(pos_cnt * test_frac)
    print(f"\n* 참고: 만약 Test Set이 {test_frac*100:.0f}% 분할 기준이라면:")
    print(f"  - Test Set 전체 수량 : {test_total:,} 건")
    print(f"  - Test Set 내 1의 수량: {test_pos:,} 건  (Test Set의 TP + FN 기준)")

    # 3. Top-K 타겟팅 축(K=500, 1000, 1500, 2000, 2500, 3000) 한계점 시뮬레이션
    print("\n" + "=" * 60)
    print(f"   [Top-K 타겟팅 시뮬레이션 (전체 1의 수량: {pos_cnt:,}개 기준)]")
    print("=" * 60)
    k_list = [500, 1000, 1500, 2000, 2500, 3000]
    
    for k in k_list:
        max_possible_tp = min(k, pos_cnt)
        min_unavoidable_fp = k - max_possible_tp
        max_recall = (max_possible_tp / pos_cnt) * 100
        ideal_prec = (max_possible_tp / k) * 100
        
        print(f"[Top-{k:4d}] 이론상 최대 TP: {max_possible_tp:4d} | 불가피한 최소 FP: {min_unavoidable_fp:4d} | "
              f"최대 Recall: {max_recall:6.2f}% | 정밀도(Prec): {ideal_prec:6.2f}%")
    print("=" * 60)


[로드 중...] /mnt/d/LJH/data/final_data_100k_64.parquet

           [전체 데이터셋 Target(1) 분포 확인]
전체 샘플 수 (Total) :    9,999,329 건
실제 양성 수 (1, Pos) :       12,340 건  (전체 TP + FN)
실제 음성 수 (0, Neg) :    9,986,989 건  (전체 TN + FP)
양성(1) 비율          :       0.1234 %

* 참고: 만약 Test Set이 15% 분할 기준이라면:
  - Test Set 전체 수량 : 1,499,899 건
  - Test Set 내 1의 수량: 1,851 건  (Test Set의 TP + FN 기준)

   [Top-K 타겟팅 시뮬레이션 (전체 1의 수량: 12,340개 기준)]
[Top- 500] 이론상 최대 TP:  500 | 불가피한 최소 FP:    0 | 최대 Recall:   4.05% | 정밀도(Prec): 100.00%
[Top-1000] 이론상 최대 TP: 1000 | 불가피한 최소 FP:    0 | 최대 Recall:   8.10% | 정밀도(Prec): 100.00%
[Top-1500] 이론상 최대 TP: 1500 | 불가피한 최소 FP:    0 | 최대 Recall:  12.16% | 정밀도(Prec): 100.00%
[Top-2000] 이론상 최대 TP: 2000 | 불가피한 최소 FP:    0 | 최대 Recall:  16.21% | 정밀도(Prec): 100.00%
[Top-2500] 이론상 최대 TP: 2500 | 불가피한 최소 FP:    0 | 최대 Recall:  20.26% | 정밀도(Prec): 100.00%
[Top-3000] 이론상 최대 TP: 3000 | 불가피한 최소 FP:    0 | 최대 Recall:  24.31% | 정밀도(Prec): 100.00%
